# 00 — Tải bộ dữ liệu UCI vào Google Drive

**Chỉ chạy một lần trước P1.** Tải trực tiếp từ UCI vào thư mục đã tạo `MyDrive/FJCAI2027_PCRR/data/raw/`. Không cần đổi đường dẫn. Chỉ dùng dữ liệu gốc, không chia tập và không huấn luyện.

Nguồn gốc: [Dry Bean](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset), [Rice](https://archive.ics.uci.edu/dataset/545/rice+cammeo+and+osmancik), [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing). Các bộ này được UCI ghi giấy phép CC BY 4.0. Trích dẫn nguồn trong bài báo.

**Mục tiêu:** `dry_bean.arff` (13.611 dòng), `rice.arff` (3.810 dòng), `bank-additional-full.csv` (41.188 dòng), các ZIP gốc và tệp `dataset_manifest.json` có SHA256.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FJCAI2027_PCRR')
RAW = PROJECT_ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
(RAW / '_archives').mkdir(exist_ok=True)
print('RAW =', RAW)
assert RAW.is_dir(), 'Không thể truy cập data/raw trên Drive'


Mounted at /content/drive
RAW = /content/drive/MyDrive/FJCAI2027_PCRR/data/raw


In [ ]:
import io, os, json, hashlib, shutil, zipfile, requests
from datetime import datetime, timezone
from urllib.parse import quote

DATASETS = {
 'dry_bean': {'uci_id':602, 'zip':'dry_bean_uci602.zip', 'url':'https://archive.ics.uci.edu/static/public/602/dry%2Bbean%2Bdataset.zip', 'name':'dry_bean.arff', 'rows':13611, 'features':16, 'classes':7},
 'rice': {'uci_id':545, 'zip':'rice_uci545.zip', 'url':'https://archive.ics.uci.edu/static/public/545/rice%2Bcammeo%2Band%2Bosmancik.zip', 'name':'rice.arff', 'rows':3810, 'features':7, 'classes':2},
 'bank_marketing': {'uci_id':222, 'zip':'bank_marketing_uci222.zip', 'url':'https://archive.ics.uci.edu/static/public/222/bank%2Bmarketing.zip', 'name':'bank-additional-full.csv', 'rows':41188, 'features':20, 'classes':2},
}

def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for block in iter(lambda: f.read(1<<20), b''): h.update(block)
 return h.hexdigest()

def download_zip(url, dest):
 if dest.exists() and zipfile.is_zipfile(dest):
  print('Dùng ZIP đã có:', dest.name); return
 print('Đang tải:', url)
 tmp = dest.with_suffix('.partial')
 try:
  with requests.get(url, timeout=(25,180), stream=True, headers={'User-Agent':'Mozilla/5.0 (academic reproducibility)'}) as r:
   r.raise_for_status()
   with open(tmp,'wb') as f:
    for chunk in r.iter_content(1<<18):
     if chunk: f.write(chunk)
  if not zipfile.is_zipfile(tmp): raise ValueError('Phản hồi không phải ZIP hợp lệ')
  tmp.replace(dest)
 except Exception:
  tmp.unlink(missing_ok=True)
  raise

def get_nested_bytes(zf, suffix, depth=0):
 """Tìm file trong ZIP UCI, kể cả ZIP lồng nhau."""
 candidates=[n for n in zf.namelist() if n.lower().endswith(suffix.lower())]
 if candidates:
  if len(candidates)>1: print('Nhiều tệp phù hợp:', candidates)
  return zf.read(candidates[0])
 if depth >= 3: return None
 for n in zf.namelist():
  if n.lower().endswith('.zip'):
   try:
    with zipfile.ZipFile(io.BytesIO(zf.read(n))) as inner:
     result=get_nested_bytes(inner,suffix,depth+1)
     if result is not None: return result
   except (zipfile.BadZipFile, OSError):
    continue
 return None

def extract_expected(zip_path, kind, output):
 target={'dry_bean':'dry_bean_dataset.arff',
         'rice':'rice_cammeo_osmancik.arff',
         'bank_marketing':'bank-additional-full.csv'}[kind]
 with zipfile.ZipFile(zip_path) as zf:
  content=get_nested_bytes(zf,target)
  if content is None and kind=='rice':
   # Một số bản lưu trữ đặt tên ARFF là Rice.arff.
   content=get_nested_bytes(zf,'rice.arff')
  if content is None:
   raise FileNotFoundError(f'Không tìm thấy {target} trong {zip_path}; kiểm tra cấu trúc ZIP UCI')
  with open(output,'wb') as f: f.write(content)
 return len(content)

manifest = {'created_at_utc':datetime.now(timezone.utc).isoformat(), 'source':'UCI Machine Learning Repository','license':'CC BY 4.0', 'datasets':{}}
for key, meta in DATASETS.items():
 archive=RAW/'_archives'/meta['zip']; out=RAW/meta['name']
 download_zip(meta['url'], archive)
 if not out.exists() or out.stat().st_size==0:
  extract_expected(archive,key,out)
 manifest['datasets'][key]={
  'uci_id':meta['uci_id'], 'source_url':meta['url'], 'expected_rows':meta['rows'], 'expected_features':meta['features'], 'expected_classes':meta['classes'],
  'raw_file':meta['name'], 'archive':'_archives/'+meta['zip'], 'sha256':sha256_file(out), 'archive_sha256':sha256_file(archive), 'bytes':out.stat().st_size
 }
 print('OK',key,out.name, out.stat().st_size,'bytes')


Đang tải: https://archive.ics.uci.edu/static/public/602/dry%2Bbean%2Bdataset.zip
OK dry_bean dry_bean.arff 3813205 bytes
Đang tải: https://archive.ics.uci.edu/static/public/545/rice%2Bcammeo%2Band%2Bosmancik.zip
OK rice rice.arff 427635 bytes
Đang tải: https://archive.ics.uci.edu/static/public/222/bank%2Bmarketing.zip
OK bank_marketing bank-additional-full.csv 5834924 bytes


In [ ]:
import pandas as pd
from scipy.io import arff

for key, meta in DATASETS.items():
 target=RAW/meta['name']
 if key=='bank_marketing':
  frame=pd.read_csv(target,sep=';')
  assert 'y' in frame.columns, f'{key}: thiếu y'
  X=frame.drop(columns=['y']); y=frame['y']
 else:
  data,_=arff.loadarff(target)
  frame=pd.DataFrame(data)
  label_col='Class' if 'Class' in frame.columns else 'class'
  assert label_col in frame.columns, f'{key}: thiếu Class'
  X=frame.drop(columns=[label_col]); y=frame[label_col]
 nrows,nfeat=len(frame),X.shape[1]
 nclass=y.nunique(dropna=False)
 assert nrows==meta['rows'],f'{key}: số dòng sai {nrows}'
 assert nfeat==meta['features'],f'{key}: số đặc trưng sai {nfeat}'
 assert nclass==meta['classes'],f'{key}: số lớp sai {nclass}'
 manifest['datasets'][key]['verified_shape']=[nrows,nfeat]
 manifest['datasets'][key]['verified_classes']=nclass
 print(f'PASS {key}: {nrows} mẫu, {nfeat} đặc trưng, {nclass} lớp')

with open(RAW/'dataset_manifest.json','w',encoding='utf-8') as f:
 json.dump(manifest,f,ensure_ascii=False,indent=2)
print('Đã lưu manifest:', RAW/'dataset_manifest.json')
print('DATASET_DOWNLOAD_PASS: 3/3')


PASS dry_bean: 13611 mẫu, 16 đặc trưng, 7 lớp
PASS rice: 3810 mẫu, 7 đặc trưng, 2 lớp
PASS bank_marketing: 41188 mẫu, 20 đặc trưng, 2 lớp
Đã lưu manifest: /content/drive/MyDrive/FJCAI2027_PCRR/data/raw/dataset_manifest.json
DATASET_DOWNLOAD_PASS: 3/3


## Nghiệm thu trước P1

Chỉ tiếp tục P1 khi thấy `DATASET_DOWNLOAD_PASS: 3/3` và các tệp xuất hiện trong thư mục Drive. Nếu Colab bị ngắt kết nối, chạy lại từ đầu; ZIP hợp lệ đã có sẽ được tái sử dụng.

**Không biến đổi dữ liệu gốc.** P1 phải ghi processed data sang `data/processed`, tách `data/splits`, không ghi đè `data/raw`. Thông số và độ tin cậy phải được khóa từ P1–P4 trước khi mở Test ở P5.
